# Morris Subbasin analysis

### 1. File header read

In [3]:
import pandas as pd

# 작업 경로
path = r"C:\Users\sl177\OneDrive - University of Illinois - Urbana\02_Paper Work\01_Paper(CANOPY)\01_SWAT Uncertainty\SWAT-GUSA-Official\Morris\0224_Morris subbasin analysis"

file1 = path + r"\1_Morris_prmtr_info_swapper.csv"
file2 = path + r"\2_Morris_results.csv"
file3 = path + r"\3_subbasin_group.csv"

# 파일 불러오기
df1 = pd.read_csv(file1)
df2 = pd.read_csv(file2)
df3 = pd.read_csv(file3)

print("===== File 1 =====")
print("Columns:", df1.columns.tolist())
print(df1.head(9))
print("\n")

print("===== File 2 =====")
print("Columns:", df2.columns.tolist())
print(df2.head(7))
print("\n")

print("===== File 3 =====")
print("Columns:", df3.columns.tolist())
print(df3.head(7))

===== File 1 =====
Columns: ['SWAT Model Parameter replacement', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13', 'Unnamed: 14', 'Unnamed: 15', 'Unnamed: 16', 'Unnamed: 17', 'Unnamed: 18', 'Unnamed: 19', 'Unnamed: 20', 'Unnamed: 21', 'Unnamed: 22', 'Unnamed: 23', 'Unnamed: 24', 'Unnamed: 25', 'Unnamed: 26', 'Unnamed: 27', 'Unnamed: 28', 'Unnamed: 29', 'Unnamed: 30', 'Unnamed: 31', 'Unnamed: 32', 'Unnamed: 33', 'Unnamed: 34', 'Unnamed: 35', 'Unnamed: 36', 'Unnamed: 37', 'Unnamed: 38', 'Unnamed: 39', 'Unnamed: 40', 'Unnamed: 41', 'Unnamed: 42', 'Unnamed: 43', 'Unnamed: 44', 'Unnamed: 45', 'Unnamed: 46', 'Unnamed: 47', 'Unnamed: 48', 'Unnamed: 49', 'Unnamed: 50', 'Unnamed: 51', 'Unnamed: 52', 'Unnamed: 53', 'Unnamed: 54', 'Unnamed: 55', 'Unnamed: 56', 'Unnamed: 57', 'Unnamed: 58', 'Unnamed: 59', 'Unnamed: 60', 'Unnamed: 61', 'Unnamed: 62', 'Unnamed: 63',

### 2. Parameter grouping

In [5]:
import pandas as pd
import numpy as np

path = r"C:\Users\sl177\OneDrive - University of Illinois - Urbana\02_Paper Work\01_Paper(CANOPY)\01_SWAT Uncertainty\SWAT-GUSA-Official\Morris\0224_Morris subbasin analysis"

file1 = path + r"\1_Morris_prmtr_info_swapper.csv"
file3 = path + r"\3_subbasin_group.csv"

# ---------------------------
# 1️⃣ File1 처리
# ---------------------------

df1 = pd.read_csv(file1, header=2)  # IDX가 헤더

records = []

for _, row in df1.iterrows():
    idx = row["IDX"]
    par = f"par{int(idx)}"
    
    # subbasin 값들은 4번째 컬럼부터
    sub_values = row.iloc[3:].dropna()
    sub_values = sub_values[sub_values != -9999]
    
    if len(sub_values) == 0:
        records.append([par, "A"])
    else:
        for sb in sub_values:
            records.append([par, int(sb)])

df1_long = pd.DataFrame(records, columns=["Par", "Subbasin"])

# ---------------------------
# 2️⃣ File3 처리
# ---------------------------

df3 = pd.read_csv(file3)

group_records = []

for _, row in df3.iterrows():
    group = row["Group"]
    
    # 숫자 group만 사용 (S1,S2,A 제외)
    if not str(group).isdigit():
        continue
        
    sub_values = row.iloc[1:].dropna()
    
    for sb in sub_values:
        group_records.append([int(sb), group])

df3_long = pd.DataFrame(group_records, columns=["Subbasin", "Group"])

# ---------------------------
# 3️⃣ merge
# ---------------------------

merged = df1_long.merge(df3_long, on="Subbasin", how="left")

# subbasin 있는 parameter는 group 숫자
# subbasin 없는 parameter는 이미 A
merged["Group"] = merged["Group"].fillna("A")

final = merged[["Par", "Group"]].drop_duplicates()

final.to_csv(path + r"\Par_Group_Output.csv", index=False)

print(final.head(15))
print("Total parameters:", final["Par"].nunique())

       Par Group
0     par1     A
1     par2     A
2     par3     A
3     par4     A
4     par5     A
5     par6    19
20    par7    19
35    par8    19
50    par9    19
65   par10    19
80   par11    19
95   par12    21
100  par13    21
105  par14    21
110  par15    21
Total parameters: 236


### 3. Morris results replace to group

In [8]:
import pandas as pd
import numpy as np

path = r"C:\Users\sl177\OneDrive - University of Illinois - Urbana\02_Paper Work\01_Paper(CANOPY)\01_SWAT Uncertainty\SWAT-GUSA-Official\Morris\0224_Morris subbasin analysis"

file2 = path + r"\2_Morris_results.csv"
par_group_file = path + r"\Par_Group_Output.csv"

# ---------------------------
# 1️⃣ par → group 매핑 불러오기
# ---------------------------

par_group = pd.read_csv(par_group_file)
par_group_dict = dict(zip(par_group["Par"], par_group["Group"]))

# ---------------------------
# 2️⃣ File2 불러오기
# ---------------------------

df2 = pd.read_csv(file2)

# par 컬럼들 (ParNums부터 끝까지)
par_cols = df2.columns[5:]

# ---------------------------
# 3️⃣ par → group 치환
# ---------------------------

df2_converted = df2.copy()

for col in par_cols:
    df2_converted[col] = df2_converted[col].map(par_group_dict)

# ---------------------------
# 4️⃣ 저장
# ---------------------------

save_path = path + r"\2_Morris_results_GroupReplaced.csv"
df2_converted.to_csv(save_path, index=False)

print("변환 완료")
print(df2_converted.head())

변환 완료
   Year Type            Method                               Group  Count  \
0  2011  avg  Quantile70_Group  High importance & High interaction     23   
1  2011  avg  Quantile70_Group   High importance & Low interaction      8   
2  2012  avg  Quantile70_Group  High importance & High interaction     26   
3  2012  avg  Quantile70_Group   High importance & Low interaction      6   
4  2013  avg  Quantile70_Group  High importance & High interaction     25   

  ParNums Unnamed: 6 Unnamed: 7 Unnamed: 8 Unnamed: 9  ... Unnamed: 22  \
0       5          5          7          8          8  ...           2   
1       8         17          1          3          2  ...         NaN   
2       5          5          7          8         10  ...          10   
3       1          3          4          8         10  ...         NaN   
4       5          5          7          8         10  ...           8   

  Unnamed: 23 Unnamed: 24 Unnamed: 25 Unnamed: 26 Unnamed: 27 Unnamed: 28  \
0        